In [10]:
import os
import glob
import shutil
import random
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import torchvision
from torchvision import transforms, models
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_curve, auc
import seaborn as sns
import numpy as np



In [ ]:
import gdown

FILE_ID = "1zyPVaSdmOqUDlb5Ua-bB2gl8tYDKTu2r"  

gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", "/kaggle/working/split_data.zip", quiet=False)
import zipfile

with zipfile.ZipFile("/kaggle/working/split_data.zip", "r") as zip_ref:
    zip_ref.extractall("/kaggle/working/")

print("Extraction complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1zyPVaSdmOqUDlb5Ua-bB2gl8tYDKTu2r
From (redirected): https://drive.google.com/uc?id=1zyPVaSdmOqUDlb5Ua-bB2gl8tYDKTu2r&confirm=t&uuid=450c5f8a-5253-496f-99a6-b4cd97bc63e0
To: /kaggle/working/split_data.zip
100%|██████████| 5.61G/5.61G [00:47<00:00, 118MB/s] 


Extraction complete!


In [2]:
ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB  = 'COLAB_GPU' in os.environ or os.path.exists('/content')
print(f'Running on Kaggle: {ON_KAGGLE} | Colab: {ON_COLAB}')

if ON_KAGGLE:
    AIREAL_AI_SRC   = '/kaggle/input/ai-real-art-dataset/AiArtData/AiArtData'
    AIREAL_REAL_SRC = '/kaggle/input/ai-real-art-dataset/RealArt/RealArt'
    CONTENT_AI_SRC  = '/kaggle/input/content-fake-real-dataset/fake_images'
    CONTENT_REAL_SRC= '/kaggle/input/content-fake-real-dataset/real_images'
    WORKING_DIR     = '/kaggle/working'
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    AIREAL_AI_SRC   = '/content/drive/MyDrive/AiArtData/AiArtData'
    AIREAL_REAL_SRC = '/content/drive/MyDrive/RealArt/RealArt'
    CONTENT_AI_SRC  = '/content/drive/MyDrive/fake_images'
    CONTENT_REAL_SRC= '/content/drive/MyDrive/real_images'
    WORKING_DIR     = '/content'
else:
    AIREAL_AI_SRC   = r'E:\\M.C.A\\Project\\AI-Real\\AiArtData\\AiArtData'
    AIREAL_REAL_SRC = r'E:\\M.C.A\\Project\\AI-Real\\RealArt\\RealArt'
    CONTENT_AI_SRC  = r'E:\\M.C.A\\Project\\content\\fake_images'
    CONTENT_REAL_SRC= r'E:\\M.C.A\\Project\\content\\real_images'
    WORKING_DIR     = os.getcwd()

TARGET          = os.path.join(WORKING_DIR, 'data_combined')
MODEL_SAVE_PATH = os.path.join(WORKING_DIR, 'best_vit_v10.pth')
SPLITS          = {'train': 0.7, 'val': 0.15, 'test': 0.15}

batch_size         = 32        
num_classes        = 2
base_lr            = 1e-5      
head_lr            = 1e-3      
EPOCHS             = 10        
ACCUM_STEPS        = 4         

MAX_TRAIN_SAMPLES = 50_000   
MAX_VAL_SAMPLES   = 10_000

NUM_WORKERS = 2
USE_AMP = True


Running on Kaggle: False | Colab: False


In [4]:
def collect_images(src_dir, recursive=False):
    exts = ('.jpg', '.jpeg', '.png')
    if not os.path.isdir(src_dir):
        print(f'  [WARNING] Source directory not found, skipping: {src_dir}')
        return []
    if recursive:
        paths = []
        for root, _, files in os.walk(src_dir):
            for f in files:
                if f.lower().endswith(exts):
                    paths.append(os.path.join(root, f))
        return paths
    else:
        return [os.path.join(src_dir, f)
                for f in os.listdir(src_dir)
                if f.lower().endswith(exts)]


def prepare_dataset():
    for split in SPLITS:
        for cls in ['ai', 'real']:
            os.makedirs(os.path.join(TARGET, split, cls), exist_ok=True)

    ai_train_dir = os.path.join(TARGET, 'train', 'ai')
    if os.path.exists(ai_train_dir) and len(os.listdir(ai_train_dir)) > 0:
        print('Combined dataset already prepared. Skipping copy step.')
        return

    print('Collecting AI images …')
    ai_paths   = collect_images(AIREAL_AI_SRC,   recursive=False)
    ai_paths  += collect_images(CONTENT_AI_SRC,  recursive=True)

    print('Collecting Real images …')
    real_paths  = collect_images(AIREAL_REAL_SRC, recursive=False)
    real_paths += collect_images(CONTENT_REAL_SRC, recursive=False)

    print(f'Total AI images   : {len(ai_paths):,}')
    print(f'Total Real images : {len(real_paths):,}')

    if len(ai_paths) == 0 or len(real_paths) == 0:
        raise RuntimeError('No images found! Check your dataset paths.')

    def copy_split(paths, class_name):
        random.shuffle(paths)
        n  = len(paths)
        t1 = int(n * SPLITS['train'])
        t2 = int(n * (SPLITS['train'] + SPLITS['val']))
        split_map = {'train': paths[:t1], 'val': paths[t1:t2], 'test': paths[t2:]}
        for split, files in split_map.items():
            out_dir = os.path.join(TARGET, split, class_name)
            for i, src in enumerate(files):
                ext = os.path.splitext(src)[1].lower()
                shutil.copy(src, os.path.join(out_dir, f'{i:07d}{ext}'))
        counts = {s: len(v) for s, v in split_map.items()}
        print(f'  {class_name.upper()}: train={counts["train"]:,}  val={counts["val"]:,}  test={counts["test"]:,}')

    print('\nCopying AI images …')
    copy_split(ai_paths, 'ai')
    print('Copying Real images …')
    copy_split(real_paths, 'real')
    print('\nDataset preparation complete.')


In [3]:
random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32   = True
    torch.backends.cudnn.allow_tf32         = True
    torch.backends.cudnn.benchmark          = True


Using device: cuda
  GPU: NVIDIA GeForce RTX 2050


In [6]:
prepare_dataset()

train_transforms = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply([v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05)], p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize(256), # Resize slightly larger
    v2.CenterCrop(224), # Then crop the center
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

_pin = device.type == 'cuda'
_loader_kw = dict(num_workers=2, pin_memory=_pin, persistent_workers=False)

train_dataset_full = ImageFolder(os.path.join(TARGET, 'train'), transform=train_transforms)
val_dataset_full   = ImageFolder(os.path.join(TARGET, 'val'),   transform=val_test_transforms)
test_dataset       = ImageFolder(os.path.join(TARGET, 'test'),  transform=val_test_transforms)

def balanced_subset(dataset, max_samples):
    if max_samples is None: return dataset
    targets = np.array(dataset.targets)
    per_class = max_samples // 2
    indices = []
    for cls in range(len(dataset.classes)):
        cls_idx = np.where(targets == cls)[0]
        np.random.shuffle(cls_idx)
        indices.extend(cls_idx[:per_class].tolist())
    random.shuffle(indices)
    return Subset(dataset, indices)

train_dataset = balanced_subset(train_dataset_full, MAX_TRAIN_SAMPLES)
val_dataset   = balanced_subset(val_dataset_full,   MAX_VAL_SAMPLES)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, **_loader_kw)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, **_loader_kw)

print(f'Train samples: {len(train_dataset):,} | Val samples: {len(val_dataset):,}')


Combined dataset already prepared. Skipping copy step.
Train samples: 50,000 | Val samples: 10,000


In [ ]:
print("Loading Pre-trained ViT-B/16...")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = True

model.conv_proj.requires_grad_(False)
model.class_token.requires_grad_(False)
model.encoder.pos_embedding.requires_grad_(False)


in_features = model.heads.head.in_features

model.heads.head = nn.Sequential(
    nn.Dropout(p=0.5), # Randomly zeroes 50% of the features during training
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

import platform
_compile_backend = 'inductor' if platform.system() != 'Windows' else 'eager'
try:
    model = torch.compile(model, backend=_compile_backend)
    print(f'torch.compile enabled (backend={_compile_backend})')
except Exception as e:
    print(f'torch.compile failed, using eager mode')


def get_llrd_optimizer(model, head_lr=1e-3, base_lr=1e-5, base_wd=0.05, head_wd=0.2, layer_decay=0.75):
    param_groups = []
    

    param_groups.append({
        'params': [p for p in model.heads.parameters() if p.requires_grad],
        'lr': head_lr,
        'weight_decay': head_wd 
    })
    
    num_layers = len(model.encoder.layers)
    
    for i, layer in enumerate(model.encoder.layers):
        layer_params = [p for p in layer.parameters() if p.requires_grad]
        # Only add to optimizer if the layer has unfrozen parameters!
        if layer_params:
            # Calculate decaying LR
            lr = base_lr * (layer_decay ** (num_layers - 1 - i))
            param_groups.append({
                'params': layer_params,
                'lr': lr,
                'weight_decay': base_wd # Fixed variable name
            })
        

    embed_lr = base_lr * (layer_decay ** num_layers)
    embed_params = [p for p in [model.class_token, model.encoder.pos_embedding] if p.requires_grad]
    if embed_params:
        param_groups.append({
            'params': embed_params,
            'lr': embed_lr,
            'weight_decay': 0.0
        })
        
    proj_params = [p for p in model.conv_proj.parameters() if p.requires_grad]
    if proj_params:
        param_groups.append({
            'params': proj_params,
            'lr': embed_lr,
            'weight_decay': base_wd # Fixed variable name
        })
    
    return torch.optim.AdamW(param_groups)


optimizer = get_llrd_optimizer(
    model, 
    head_lr=1e-3, 
    base_lr=3e-5,     
    base_wd=0.05, 
    head_wd=0.2,
    layer_decay=0.85   
)
warmup_epochs = 2
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - warmup_epochs)
    ],
    milestones=[warmup_epochs]
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')

ai_count    = len(os.listdir(os.path.join(TARGET, 'train', 'ai')))
real_count  = len(os.listdir(os.path.join(TARGET, 'train', 'real')))
total_count = ai_count + real_count
class_weights = torch.tensor([total_count / (2 * ai_count), total_count / (2 * real_count)], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
PATIENCE = 5

Loading Pre-trained ViT-B/16...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 216MB/s]  


torch.compile enabled (backend=inductor)


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler, accum_steps=1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum_steps * images.size(0)
        
        # Reverted back to standard 1D label matching
        _, predicted  = torch.max(outputs.detach(), 1)
        correct += (predicted == labels).sum().item()
        total   += labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f'  Batch [{batch_idx+1}/{len(dataloader)}] Loss: {running_loss/total:.4f}  Acc: {correct/total:.4f}')

    return running_loss / total, correct / total
    
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                outputs = model(images)
                loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total

def test_accuracy_tta(model, dataloader, device, threshold=0.60):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                # Original image
                outputs_orig = model(images)
                
                # Flipped image
                images_flipped = torch.flip(images, dims=[3]) 
                outputs_flipped = model(images_flipped)
                
                # Average the logits
                outputs_avg = (outputs_orig + outputs_flipped) / 2.0
                
                # Convert logits to probabilities
                probs = F.softmax(outputs_avg, dim=1)
                
                # Shift threshold: Predict 'Real' (1) only if confidence >= threshold, else 'AI' (0)
                predicted = (probs[:, 1] >= threshold).long()
                
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
    return correct / total

In [9]:
best_val_acc  = 0.0
train_losses, val_losses, train_accs, val_accs = [], [], [], []
patience_counter = 0

print('Starting training …')
for epoch in range(EPOCHS):
    t0 = time.time()
    
    print(f'\n── Epoch [{epoch+1}/{EPOCHS}] ──────────────────────────────')

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler, ACCUM_STEPS)
    val_loss,   val_acc   = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc);    val_accs.append(val_acc)

    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.2e} | Time: {elapsed:.1f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        state = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
        torch.save(state, MODEL_SAVE_PATH)
        patience_counter = 0
        print(f'  => Best model saved  (val_acc={best_val_acc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break


Starting training …

── Epoch [1/10] ──────────────────────────────


W0424 12:18:58.759000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  Batch [50/1562] Loss: 0.7547  Acc: 0.4956
  Batch [100/1562] Loss: 0.7330  Acc: 0.5088
  Batch [150/1562] Loss: 0.7214  Acc: 0.5183
  Batch [200/1562] Loss: 0.7094  Acc: 0.5309
  Batch [250/1562] Loss: 0.7019  Acc: 0.5417
  Batch [300/1562] Loss: 0.6970  Acc: 0.5471
  Batch [350/1562] Loss: 0.6936  Acc: 0.5512
  Batch [400/1562] Loss: 0.6902  Acc: 0.5555
  Batch [450/1562] Loss: 0.6853  Acc: 0.5603
  Batch [500/1562] Loss: 0.6823  Acc: 0.5640
  Batch [550/1562] Loss: 0.6788  Acc: 0.5692
  Batch [600/1562] Loss: 0.6748  Acc: 0.5735
  Batch [650/1562] Loss: 0.6731  Acc: 0.5753
  Batch [700/1562] Loss: 0.6705  Acc: 0.5788
  Batch [750/1562] Loss: 0.6674  Acc: 0.5835
  Batch [800/1562] Loss: 0.6649  Acc: 0.5871
  Batch [850/1562] Loss: 0.6622  Acc: 0.5897
  Batch [900/1562] Loss: 0.6601  Acc: 0.5916
  Batch [950/1562] Loss: 0.6581  Acc: 0.5938
  Batch [1000/1562] Loss: 0.6565  Acc: 0.5953
  Batch [1050/1562] Loss: 0.6543  Acc: 0.5979
  Batch [1100/1562] Loss: 0.6523  Acc: 0.5999
  Batch 

In [ ]:
num_classes = 2
eval_model = models.vit_b_16(weights=None)
in_feats = eval_model.heads.head.in_features
eval_model.heads.head = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_feats, num_classes))
eval_model = eval_model.to(device)
eval_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=True))
eval_model.eval()
print("Model loaded.")

# ─── 2. Pool val + test folders (both held-out, never trained on) ────────────
xfm = v2.Compose([
    v2.ToImage(), v2.Resize(256), v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_ds  = ImageFolder(os.path.join(TARGET, 'val'),  transform=xfm)
test_ds = ImageFolder(os.path.join(TARGET, 'test'), transform=xfm)

pool = ConcatDataset([val_ds, test_ds])
pool_targets = np.array(val_ds.targets + test_ds.targets)

rng     = np.random.default_rng(42)
ai_idx  = np.where(pool_targets == 0)[0]
real_idx= np.where(pool_targets == 1)[0]
per_cls = min(len(ai_idx), len(real_idx), 31_333)   

ai_sel   = rng.choice(ai_idx,   size=per_cls, replace=False)
real_sel = rng.choice(real_idx, size=per_cls, replace=False)
bal_idx  = np.concatenate([ai_sel, real_sel])
rng.shuffle(bal_idx)

eval_dataset = Subset(pool, bal_idx.tolist())
eval_loader  = DataLoader(eval_dataset, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=(device.type == 'cuda'))

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in eval_loader:
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits_avg = (eval_model(images) + eval_model(torch.flip(images, dims=[3]))) / 2.0
            probs      = F.softmax(logits_avg, dim=1)
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc    = auc(fpr, tpr)
best_idx   = np.argmax(tpr - fpr)
opt_thresh = thresholds[best_idx]


Model loaded.


In [9]:
class_names = val_ds.classes   
for label, thresh in [(f"Optimal {opt_thresh:.3f}", opt_thresh)]:
    preds = (all_probs >= thresh).astype(int)
    print(f"\n{'='*57}")
    print("Vision Transformer")
    print(f"{'='*57}")
    print(classification_report(all_labels, preds, target_names=class_names, digits=4))

final_preds = (all_probs >= opt_thresh).astype(int)
cm = confusion_matrix(all_labels, final_preds)
cm_path = os.path.join(WORKING_DIR, 'confusion_matrix_v10_balanced1.png')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion Matrix — ViT')
plt.tight_layout()
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"\nConfusion Matrix {cm_path}")


Vision Transformer
              precision    recall  f1-score   support

          ai     0.9049    0.8870    0.8958     31333
        real     0.8892    0.9067    0.8979     31333

    accuracy                         0.8969     62666
   macro avg     0.8970    0.8969    0.8969     62666
weighted avg     0.8970    0.8969    0.8969     62666


Confusion Matrix e:\M.C.A\Project\backend\VIT\confusion_matrix_v10_balanced1.png
